# ChurnLens - Exploratory Data Analysis (EDA)
**Dataset:** IBM Telco Customer Churn (7,043 rows, 33 columns)  
**Objective:** Uncover key churn drivers, quantify contract hazard rates, analyze tenure retention curves, and identify target leakage risks before feature pipeline design.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 100

data_path = Path('../data/raw/Telco_customer_churn.csv')
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")


## 1. Churn Distribution & Class Imbalance
We examine the distribution of `Churn Value` (0 vs 1).


In [ ]:
churn_counts = df['Churn Value'].value_counts()
churn_rates = df['Churn Value'].value_counts(normalize=True) * 100

print(f"Retained (0): {churn_counts[0]:,} ({churn_rates[0]:.2f}%)")
print(f"Churned  (1): {churn_counts[1]:,} ({churn_rates[1]:.2f}%)")
print(f"Imbalance Ratio: {churn_counts[0] / churn_counts[1]:.2f} : 1")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Retained (0)', 'Churned (1)'], churn_counts.values, color=['#2563eb', '#dc2626'], width=0.5)
ax.set_ylabel('Number of Customers')
ax.set_title('Churn Class Distribution')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f"{yval:,} ({yval/len(df)*100:.1f}%)", ha='center', va='bottom')
plt.tight_layout()
plt.show()


## 2. Data Cleaning & Type Anomalies
`Total Charges` is stored as `object` because 11 rows have whitespace `" "` strings. All 11 correspond to new accounts with `Tenure Months == 0`.


In [ ]:
blank_tc = df[df['Total Charges'].str.strip() == '']
print(f"Number of blank Total Charges records: {len(blank_tc)}")
print(f"Tenure distribution of blank records: {blank_tc['Tenure Months'].value_counts().to_dict()}")

# Coerce for subsequent analysis
df['Total Charges Numeric'] = pd.to_numeric(df['Total Charges'].str.strip().replace('', '0.0'), errors='coerce')


## 3. Contract Type vs. Churn Rate
Month-to-month contracts have historically exhibited the highest churn risk.


In [ ]:
contract_churn = df.groupby('Contract')['Churn Value'].agg(['count', 'mean']).reset_index()
contract_churn['mean'] = contract_churn['mean'] * 100
contract_churn.columns = ['Contract', 'Total Customers', 'Churn Rate (%)']
display(contract_churn)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=contract_churn, x='Contract', y='Churn Rate (%)', palette=['#ef4444', '#f59e0b', '#10b981'], ax=ax)
ax.set_title('Churn Rate by Contract Type')
ax.set_ylim(0, 50)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 7), textcoords='offset points')
plt.tight_layout()
plt.show()


## 4. Tenure Dynamics
Customers in their first 6 months have the highest attrition rate.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=df, x='Tenure Months', hue='Churn Label', multiple='stack', bins=36, palette={'Yes': '#ef4444', 'No': '#3b82f6'}, ax=ax)
ax.set_title('Customer Tenure Distribution by Churn Status')
ax.set_xlabel('Tenure (Months)')
plt.tight_layout()
plt.show()


## 5. Pricing Pressure: Monthly Charges
High monthly charges (especially above $70/month for Fiber Optic without tech support) correlate with elevated churn.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=df, x='Monthly Charges', hue='Churn Label', common_norm=False, fill=True, palette={'Yes': '#ef4444', 'No': '#3b82f6'}, ax=ax)
ax.set_title('Kernel Density Estimate of Monthly Charges by Churn')
ax.set_xlabel('Monthly Charges ($)')
plt.tight_layout()
plt.show()


## 6. Critical Engineering Finding: Target Leakage Quarantine
- `Churn Score`: Vendor pre-computed metric ($r = 0.665$ with churn). Must NOT be fed into ML models.
- `Churn Reason`: Available only for churned users ($N=1869$). 100% target leakage.
- `Count`, `Country`, `State`: Single unique values across all 7,043 rows (zero variance).
- `CustomerID`: High cardinality identifier, must be excluded from feature transformations.
